<a href="https://colab.research.google.com/github/nanditapandey000/Data-Visualization-and-Communication/blob/main/week5_visualization_portfolio_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 — Data Visualization Portfolio: LendingClub Loan Defaults

**Dataset:** LendingClub LoanStats3a (Week 1 cleaned, 42,535 loans, 2007–2011)
**Objective:** Build a portfolio of ten annotated visualizations that together tell the story of consumer-loan default risk — from the platform's growth, through the grade signal that dominates risk, to the financial and geographic mechanisms that compound it.

## Storyboard

The portfolio follows a five-act narrative arc:

| Act | Question | Figures |
|-----|----------|---------|
| I. The rise | How did LendingClub grow, and did defaults rise with volume? | 1, 2 |
| II. The signal | What single feature best predicts default? | 3, 6, 8 |
| III. The dimensions | How do purpose, geography, and debt burden modulate risk? | 4, 5, 7 |
| IV. Interactions | How do risk factors combine at the loan level? | 9, 10 |
| V. Synthesis | What is the default story, and what should a lender do? | narrative |

> **Reproducibility.** Upload `cleaned_dataset.csv` to the Colab working directory and Run All. All figures use a consistent editorial palette and a single random seed.

## 1. Environment setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import squarify
from scipy import stats

# Editorial palette
NAVY  = "#1d3557"
RED   = "#c1121f"
GREEN = "#386641"
GOLD  = "#f4a261"
SLATE = "#6c757d"
TEAL  = "#2a9d8f"
LIGHT_BG = "#f8f9fa"
GRID  = "#e0e0e0"
RISK_CMAP = sns.color_palette("YlOrRd", as_cmap=True)
GRADE_COLORS = sns.color_palette("RdYlGn_r", 7).as_hex()

sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlecolor"] = NAVY
plt.rcParams["axes.edgecolor"] = "#4a4a4a"
plt.rcParams["axes.labelcolor"] = "#2b2b2b"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment ready.")

## 2. Data loading and preparation

We construct the binary target `is_default` exactly as in Weeks 3 and 4 (any non-Fully-Paid status), and parse the issue date to support the timeline visualizations.

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")
df["issue_d"] = pd.to_datetime(df["issue_d"], errors="coerce")
df["issue_month_dt"] = df["issue_d"].dt.to_period("M").dt.to_timestamp()

default_statuses = ["Charged Off",
                    "Does not meet the credit policy. Status:Charged Off"]
df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)

grade_order = ["A", "B", "C", "D", "E", "F", "G"]
grade_map = {g: i+1 for i, g in enumerate(grade_order)}
df["grade_ord"] = df["grade"].map(grade_map)

print(f"Loaded {len(df):,} loans; default rate = {df['is_default'].mean()*100:.2f}%")
print(f"Date range: {df['issue_d'].min().date()} to {df['issue_d'].max().date()}")

## 3. Visualization Portfolio

### Figure 1 — Loan origination growth, 2007–2011

**Visual technique:** Dual-axis area + line chart. The area emphasizes the cumulative mass of loans; the line emphasizes the dollar trajectory. Peak annotation draws the eye to the climax of the growth story.

**Why this technique:** The two series share a time dimension but have different units (count vs dollars). Dual axes let us see both in one view; area + line makes the growth pattern immediately visible.

In [ ]:
monthly = (df.groupby("issue_month_dt")
              .agg(loan_count=("loan_amnt", "count"),
                   total_volume=("loan_amnt", "sum"))
              .reset_index())
monthly["total_volume_m"] = monthly["total_volume"] / 1e6

fig, ax1 = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
ax1.fill_between(monthly["issue_month_dt"], monthly["loan_count"],
                 color=NAVY, alpha=0.18)
ax1.plot(monthly["issue_month_dt"], monthly["loan_count"],
         color=NAVY, lw=2.2, marker="o", ms=4)
ax1.set_xlabel("Issue month"); ax1.set_ylabel("Number of loans", color=NAVY)
ax1.tick_params(axis="y", labelcolor=NAVY)
ax1.grid(axis="y", alpha=0.3, color=GRID)

ax2 = ax1.twinx()
ax2.plot(monthly["issue_month_dt"], monthly["total_volume_m"],
         color=GOLD, lw=2.2, ls="--", marker="s", ms=4)
ax2.set_ylabel("Total volume ($M)", color=GOLD)
ax2.tick_params(axis="y", labelcolor=GOLD)

peak = monthly.loc[monthly["loan_count"].idxmax()]
ax1.annotate(f"Peak: {int(peak['loan_count']):,} loans\n(${peak['total_volume_m']:.0f}M)",
             xy=(peak["issue_month_dt"], peak["loan_count"]),
             xytext=(-90, -10), textcoords="offset points",
             fontsize=9, color=NAVY, fontweight="bold",
             arrowprops=dict(arrowstyle="->", color=NAVY, lw=1.2))
ax1.set_title("LendingClub Loan Origination Volume, 2007–2011", loc="left", pad=12)
plt.show()

print(f"Peak month: {peak['issue_month_dt'].date()} with {int(peak['loan_count']):,} loans")
print(f"Total volume: ${monthly['total_volume_m'].sum():.1f}M across all loans")

### Figure 2 — Default rate by origination year

**Visual technique:** Line chart with value labels and a horizontal baseline. The baseline anchors each year's deviation from the overall average; value labels remove the need to estimate from the y-axis.

**Key insight:** Default rates were elevated for the 2008 crisis cohort and rose again toward 2011 as the platform grew — growth did not eliminate risk.

In [ ]:
yearly = (df.groupby("issue_year")["is_default"]
             .agg(["mean", "count"]).reset_index())
yearly["default_rate_pct"] = yearly["mean"] * 100
overall = df["is_default"].mean() * 100

fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
ax.plot(yearly["issue_year"], yearly["default_rate_pct"],
        color=RED, lw=2.5, marker="o", ms=9, zorder=3)
ax.fill_between(yearly["issue_year"], yearly["default_rate_pct"], color=RED, alpha=0.12)
ax.axhline(overall, color=SLATE, ls="--", lw=1.3)
ax.text(yearly["issue_year"].iloc[-1] + 0.05, overall + 0.3,
        f"Overall avg: {overall:.1f}%", color=SLATE, fontsize=9, ha="right")

for _, r in yearly.iterrows():
    ax.annotate(f"{r['default_rate_pct']:.1f}%",
                xy=(r["issue_year"], r["default_rate_pct"]),
                xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=10, fontweight="bold", color=NAVY)

ax.set_xlabel("Origination year"); ax.set_ylabel("Default rate (%)")
ax.set_xticks(yearly["issue_year"])
ax.set_ylim(0, max(yearly["default_rate_pct"]) * 1.25)
ax.grid(axis="y", alpha=0.3, color=GRID)
ax.set_title("Default Rate by Origination Year", loc="left", pad=12)
plt.show()

### Figure 3 — The grade ladder

**Visual technique:** Vertical bar chart with a diverging green-to-red colormap, ordered A→G. Annotations carry both the rate and the sample size, so the reader can weigh statistical confidence.

**Key insight:** Default rate rises monotonically from 6% (grade A) to 34% (grade G) — a 5.5× spread. Risk escalates sharply after grade D. **This is the pivot of the narrative.**

In [ ]:
g = (df.groupby("grade")["is_default"]
        .agg(["mean", "count"]).reindex(grade_order).reset_index())
g["default_rate_pct"] = g["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
bars = ax.bar(g["grade"], g["default_rate_pct"],
              color=GRADE_COLORS, edgecolor="white", lw=1.2, width=0.7)
overall = df["is_default"].mean() * 100
ax.axhline(overall, color=SLATE, ls="--", lw=1.3)
ax.text(6.4, overall + 0.3, f"Avg: {overall:.1f}%", color=SLATE, fontsize=9, ha="right")

for bar, (_, r) in zip(bars, g.iterrows()):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.4,
            f"{r['default_rate_pct']:.1f}%", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color=NAVY)
    ax.text(bar.get_x() + bar.get_width()/2, 0.5,
            f"n={int(r['count']):,}", ha="center", va="bottom",
            fontsize=8, color="white", fontweight="bold")

ax.annotate("", xy=(5, g.loc[g.grade=="F", "default_rate_pct"].iloc[0]),
            xytext=(4, g.loc[g.grade=="E", "default_rate_pct"].iloc[0]),
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.5))
ax.text(4.5, g.loc[g.grade=="E", "default_rate_pct"].iloc[0] + 4,
        "Risk escalates\nsharply after grade D",
        ha="center", fontsize=9, color=RED, style="italic")

ax.set_xlabel("Loan grade"); ax.set_ylabel("Default rate (%)")
ax.set_ylim(0, g["default_rate_pct"].max() * 1.2)
ax.grid(axis="y", alpha=0.3, color=GRID)
ax.set_title("The Grade Ladder: Default Rate Rises Monotonically with Risk Grade",
             loc="left", pad=12)
plt.show()

### Figure 4 — Loan purpose treemap

**Visual technique:** Treemap with sequential color overlay. Encodes two dimensions (volume + rate) in one view; the long tail of small purposes remains legible.

**Key insight:** Debt consolidation is ~47% of volume; small_business loans default at 28%, the highest of any purpose.

In [ ]:
p = (df.groupby("purpose")["is_default"]
        .agg(["mean", "count"]).reset_index().sort_values("count", ascending=False))
p["default_rate_pct"] = p["mean"] * 100

norm = (p["default_rate_pct"] - p["default_rate_pct"].min()) / \
       (p["default_rate_pct"].max() - p["default_rate_pct"].min() + 1e-9)
colors = [RISK_CMAP(0.25 + 0.6 * n) for n in norm]
labels = [f"{row.purpose}\n{int(row.count):,} loans\n{row.default_rate_pct:.1f}% default"
          for row in p.itertuples(index=False)]

fig, ax = plt.subplots(figsize=(12, 6.5), constrained_layout=True)
squarify.plot(sizes=p["count"], label=labels, color=colors, alpha=0.92,
              text_kwargs={"fontsize": 8.5, "color": "white", "fontweight": "bold"},
              ax=ax, pad=True)
ax.axis("off")

sm = plt.cm.ScalarMappable(cmap=RISK_CMAP,
                            norm=plt.Normalize(vmin=p["default_rate_pct"].min(),
                                               vmax=p["default_rate_pct"].max()))
fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.02).set_label("Default rate (%)", fontsize=9)
ax.set_title("Loan Purpose: Volume vs Default Rate", loc="left", pad=10,
             fontsize=14, fontweight="bold", color=NAVY)
plt.show()

print(f"Top purpose: {p.iloc[0]['purpose']} ({int(p.iloc[0]['count']):,} loans)")
print(f"Riskiest purpose: {p.loc[p['default_rate_pct'].idxmax(), 'purpose']} "
      f"({p['default_rate_pct'].max():.1f}% default)")

### Figure 5 — Default rate by US state (tile-grid map)

**Visual technique:** Tile-grid cartogram (equal-area squares). Each state gets equal visual weight; color encodes the default rate. **States with fewer than 50 loans are shown in gray** to avoid small-sample outliers (e.g. Nebraska's apparent 55% rate rests on only 11 loans).

**Key insight:** Nevada leads at 23.7% (among states with adequate sample); rates cluster higher in the Southeast/Southwest. State-level variation is far smaller than grade-level variation.

In [ ]:
s = (df.groupby("addr_state")["is_default"]
        .agg(["mean", "count"]).reset_index())
s["default_rate_pct"] = s["mean"] * 100
MIN_N = 50
s_filtered = s[s["count"] >= MIN_N].copy()

grid = {
    "AK": (0,0), "ME": (10,6), "VT": (9,5), "NH": (10,5),
    "WA": (1,4), "ID": (2,3), "MT": (3,4), "ND": (4,4),
    "MN": (5,4), "WI": (6,4), "MI": (7,4), "NY": (9,4), "MA": (10,4),
    "OR": (1,3), "UT": (2,2), "WY": (3,3), "SD": (4,3), "IA": (5,3),
    "IL": (6,3), "IN": (7,3), "OH": (8,3), "PA": (9,3), "NJ": (10,3), "CT": (11,4), "RI": (11,5),
    "CA": (1,2), "NV": (2,1), "CO": (3,2), "NE": (4,2), "MO": (5,2),
    "KY": (7,2), "WV": (8,2), "VA": (9,2), "MD": (10,2), "DE": (11,3),
    "AZ": (2,0), "NM": (3,1), "KS": (4,1), "AR": (5,1), "TN": (6,2),
    "NC": (9,1), "DC": (10,1),
    "OK": (4,0), "LA": (5,0), "MS": (6,1), "AL": (7,1), "GA": (8,1),
    "SC": (9,0), "HI": (0,1), "TX": (3,0), "FL": (9,0),
}
merged = pd.DataFrame([{"state": st, "x": x, "y": y} for st, (x, y) in grid.items()])
merged = merged.merge(s_filtered, left_on="state", right_on="addr_state", how="left")

fig, ax = plt.subplots(figsize=(13, 7), constrained_layout=True)
vmax = s_filtered["default_rate_pct"].quantile(0.95)
for _, r in merged.iterrows():
    if not pd.isna(r["default_rate_pct"]):
        color = RISK_CMAP(r["default_rate_pct"] / vmax); edge = "white"
        rate_str = f"{r['default_rate_pct']:.0f}%"; text_color = "white"
    else:
        color = "#e8e8e8"; edge = "#b0b0b0"; rate_str = "n<50"; text_color = SLATE
    rect = mpatches.FancyBboxPatch((r["x"], r["y"]), 0.9, 0.9,
            boxstyle="round,pad=0.02", lw=1.2, edgecolor=edge, facecolor=color)
    ax.add_patch(rect)
    ax.text(r["x"]+0.45, r["y"]+0.62, r["state"], ha="center", va="center",
            fontsize=9.5, fontweight="bold", color=text_color)
    ax.text(r["x"]+0.45, r["y"]+0.28, rate_str, ha="center", va="center",
            fontsize=8, color=text_color)

top_state = s_filtered.loc[s_filtered["default_rate_pct"].idxmax()]
top_grid = grid[top_state["addr_state"]]
ax.annotate(f"Highest (n≥50): {top_state['addr_state']}\n"
            f"{top_state['default_rate_pct']:.1f}% ({int(top_state['count']):,} loans)",
            xy=(top_grid[0]+0.45, top_grid[1]+0.95), xytext=(7.5, 6.5),
            fontsize=9, color=RED, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.3))

ax.set_xlim(-0.5, 12.5); ax.set_ylim(-0.5, 7.5)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.axis("off")
sm = plt.cm.ScalarMappable(cmap=RISK_CMAP, norm=plt.Normalize(vmin=0, vmax=vmax))
fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.02).set_label("Default rate (%)", fontsize=9)
ax.set_title("Default Rate by Borrower State (Tile-Grid Map)", loc="left", pad=12,
             fontsize=14, fontweight="bold", color=NAVY)
plt.show()

### Figure 6 — Interest rate tracks the grade signal

**Visual technique:** Violin plot with mean overlay. Reveals distribution shape (not just summary statistics) and the central trend simultaneously.

**Key insight:** Interest rate rises ~3–4 points per grade step, from 7.3% (A) to 20.3% (G). Dispersion widens at the risky end. LendingClub prices risk into the rate — the grade signal is operational, not just statistical.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
parts = ax.violinplot(
    [df.loc[df.grade == g, "int_rate"].dropna().values for g in grade_order],
    positions=range(len(grade_order)), showmedians=True, showextrema=False)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(GRADE_COLORS[i]); pc.set_alpha(0.7); pc.set_edgecolor("white")
parts["cmedians"].set_color(NAVY); parts["cmedians"].set_linewidth(2)

means = [df.loc[df.grade == g, "int_rate"].mean() for g in grade_order]
ax.plot(range(len(grade_order)), means, "o-", color=GOLD, ms=8, lw=2, label="Mean rate", zorder=5)

ax.set_xticks(range(len(grade_order))); ax.set_xticklabels(grade_order)
ax.set_xlabel("Loan grade"); ax.set_ylabel("Interest rate (%)")
ax.grid(axis="y", alpha=0.3, color=GRID); ax.legend(loc="upper left", fontsize=9)
ax.annotate("Each grade step ≈ +3–4%\ninterest rate",
            xy=(3, means[3]), xytext=(4.5, means[3] - 6),
            fontsize=9, color=RED, style="italic",
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.3))
ax.set_title("Interest Rate Tracks Risk Grade", loc="left", pad=12)
plt.show()

### Figure 7 — DTI distribution by outcome

**Visual technique:** Overlapping KDE plots with median markers and an inset statistical summary. KDE preserves distribution shape; the inset quantifies the effect.

**Key insight:** Defaulters carry ~2.5 DTI points more debt (median 14.3 vs 13.3), but the distributions overlap heavily — DTI is a modulator, not a determiner.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
dti_paid = df.loc[df.is_default == 0, "dti"].dropna()
dti_def = df.loc[df.is_default == 1, "dti"].dropna()
dti_paid = dti_paid[dti_paid < 50]; dti_def = dti_def[dti_def < 50]

sns.kdeplot(dti_paid, color=GREEN, fill=True, alpha=0.35, lw=2,
            label=f"Paid (n={len(dti_paid):,})", ax=ax, bw_adjust=1.1)
sns.kdeplot(dti_def, color=RED, fill=True, alpha=0.35, lw=2,
            label=f"Default (n={len(dti_def):,})", ax=ax, bw_adjust=1.1)

med_p, med_d = dti_paid.median(), dti_def.median()
ax.axvline(med_p, color=GREEN, ls="--", lw=1.5, alpha=0.7)
ax.axvline(med_d, color=RED, ls="--", lw=1.5, alpha=0.7)

t_stat, p_val = stats.ttest_ind(dti_paid, dti_def, equal_var=False)
ax.text(0.98, 0.98,
        f"Welch's t-test: t={t_stat:.2f}, p<0.001\n"
        f"Mean shift: {dti_def.mean() - dti_paid.mean():.2f} pts higher",
        transform=ax.transAxes, fontsize=9, ha="right", va="top",
        bbox=dict(boxstyle="round,pad=0.4", fc=LIGHT_BG, ec=GRID))

ax.set_xlabel("Debt-to-income ratio (DTI)"); ax.set_ylabel("Density")
ax.legend(loc="upper right", fontsize=9); ax.grid(axis="y", alpha=0.3, color=GRID)
ax.set_title("Debt-to-Income Ratio: Defaulters Carry More Debt", loc="left", pad=12)
plt.show()

### Figure 8 — Correlation matrix of numeric features

**Visual technique:** Lower-triangle correlation heatmap with annotated values and a red target marker. Diverging colormap makes positive and negative correlations symmetric and legible.

**Key insight:** Grade and interest rate dominate (r ≈ 0.20); other features contribute modestly (r ≈ 0.10). This quantifies what Figures 3 and 6 showed qualitatively.

In [ ]:
cols = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti",
        "credit_history_years", "inq_last_6mths", "open_acc",
        "revol_bal", "revol_util", "total_acc", "loan_to_income",
        "installment_to_income", "is_default", "grade_ord"]
corr = df[cols].corr()

fig, ax = plt.subplots(figsize=(11, 9), constrained_layout=True)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.7, "label": "Pearson correlation"},
            annot_kws={"size": 8}, ax=ax)

# Highlight default row
for i, c in enumerate(cols):
    if c == "is_default":
        ax.add_patch(mpatches.Rectangle((i, i), 1, 1, fill=False, edgecolor=RED, lw=2.5))
        break

ax.set_title("Correlation Matrix of Numeric Features", loc="left", pad=12,
             fontsize=14, fontweight="bold")
plt.show()

print("\nTop 5 features correlated with default:")
target_corr = corr["is_default"].drop("is_default").sort_values(ascending=False)
print(target_corr.head(5).round(3))

### Figure 9 — Loan amount vs annual income

**Visual technique:** Scatter plot with alpha-blending and color-by-outcome. Sample size limited to 6,000 for legibility; axes clipped to the distribution bulk. Annotation flags the high-risk corner.

**Key insight:** Defaults concentrate in the upper-left (high loan, low income), confirming loan_to_income as a risk driver.

In [ ]:
sample = df.sample(min(6000, len(df)), random_state=42)
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

paid = sample[sample.is_default == 0]
default = sample[sample.is_default == 1]
ax.scatter(paid["annual_inc"], paid["loan_amnt"],
           s=10, color=GREEN, alpha=0.25, label=f"Paid (n={len(paid):,})")
ax.scatter(default["annual_inc"], default["loan_amnt"],
           s=14, color=RED, alpha=0.55, label=f"Default (n={len(default):,})",
           edgecolors="white", linewidths=0.3)

ax.set_xlim(0, 250_000); ax.set_ylim(0, 40_000)
ax.annotate("High-risk corner:\nhigh loan, low income",
            xy=(30_000, 35_000), xytext=(70_000, 36_000),
            fontsize=9, color=RED, style="italic",
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.3))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"${y/1000:.0f}k"))
ax.set_xlabel("Annual income"); ax.set_ylabel("Loan amount")
ax.legend(loc="upper right", fontsize=9, framealpha=0.9); ax.grid(alpha=0.3, color=GRID)
ax.set_title("Loan Amount vs Annual Income, Colored by Outcome", loc="left", pad=12)
plt.show()

### Figure 10 — Risk matrix: purpose × grade

**Visual technique:** Two-dimensional heatmap of default rate. Both dimensions categorical and ordered; color encodes the rate. Annotation on the hottest cell highlights the interaction.

**Key insight:** Risk compounds — small_business × grade G defaults at 50%, the hottest cell. Purpose and grade are not independent. This is exactly the kind of interaction a tree-based model would exploit.

In [ ]:
top_purposes = df["purpose"].value_counts().head(8).index.tolist()
sub = df[df["purpose"].isin(top_purposes)]
mat = (sub.groupby(["purpose", "grade"])["is_default"]
         .mean().unstack().reindex(columns=grade_order))
mat_pct = (mat * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
sns.heatmap(mat_pct, annot=True, fmt=".1f", cmap=RISK_CMAP,
            cbar_kws={"label": "Default rate (%)", "shrink": 0.7},
            linewidths=0.5, linecolor="white",
            annot_kws={"size": 9, "fontweight": "bold"}, ax=ax)
ax.set_xlabel("Loan grade"); ax.set_ylabel("Loan purpose")
ax.set_title("Risk Matrix: Default Rate by Purpose × Grade", loc="left", pad=12,
             fontsize=14, fontweight="bold")
plt.show()

hottest = mat_pct.stack().idxmax()
hottest_val = mat_pct.stack().max()
print(f"\nHottest cell: {hottest[0]} × grade {hottest[1]} = {hottest_val:.1f}% default")

## 4. Conclusion

The ten visualizations together tell a coherent story:

### The headline: grade dominates
The single most important finding — surfaced in Figures 3, 6, and 8 — is that **LendingClub's assigned loan grade is the dominant predictor of default**, with a 5.5× spread between the safest and riskiest grades. This explains why logistic regression outperformed gradient-boosting ensembles: when the dominant feature is monotonic, linear models capture it efficiently.

### The modulators: purpose, geography, debt burden
Around the grade signal, three secondary dimensions modulate risk:
- **Purpose** (Figure 4): debt consolidation dominates volume; small_business and renewable_energy carry the highest purpose-specific risk.
- **Geography** (Figure 5): Nevada, Alaska, and the Southeast show elevated rates; the Northeast and Mountain West are comparatively safe.
- **Debt burden** (Figure 7): DTI shifts ~2.5 points higher for defaulters, but the overlap is substantial — DTI is a modulator, not a determiner.

### The interactions: risk compounds
The closing visualization (Figure 10) shows that risk dimensions are not independent. The small_business × grade G cell defaults at 50%, far above what either dimension would suggest in isolation.

> **The narrative in one sentence:** LendingClub grew 30× in four years, defaults remained a persistent ~15% problem, the platform's own grade signal captures most of the risk, and the remaining risk concentrates at the intersections of high-risk purposes, high-risk geographies, and high debt burdens.